In [2]:
from pathlib import Path

import pandas as pd

In [3]:
# КОНСТАНТЫ
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Не найдена директория: {DATA_DIR}")

In [ ]:
# Players
players_path = DATA_DIR / "players.csv"

if not players_path.is_file():
    raise FileNotFoundError(f"Не найден файл: {players_path}")

players = pd.read_csv(players_path)

In [5]:
players.head()

,id,registration_date,registration_type,country
0,1,2023-04-07,standard,FR
1,2,2023-02-15,premium,UK
2,3,2023-01-05,standard,DE
3,4,2023-05-11,premium,DE
4,5,2023-10-13,premium,DE


In [6]:
players.sample(n=5, random_state=42)

,id,registration_date,registration_type,country
521,522,2024-02-11,standard,US
737,738,2023-08-04,premium,US
740,741,2024-02-12,standard,FR
660,661,2023-10-22,premium,RU
411,412,2023-10-15,vip,DE


In [ ]:
print(f"СТроки: {players.shape[0]}")
print(f"Колонки: {players.shape[1]}")

СТроки: 1,000
Колонки: 4

Column names:
- id
- registration_date
- registration_type
- country


In [8]:
players.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 1000 non-null   int64
 1   registration_date  1000 non-null   str  
 2   registration_type  1000 non-null   str  
 3   country            1000 non-null   str  
dtypes: int64(1), str(3)
memory usage: 31.4 KB


In [13]:
# проверка качества данных
players_quality = pd.Series({
    "rows": len(players),
    "null_values": int(players.isna().sum().sum()),
    "duplicate_rows": int(players.duplicated().sum()),
    "duplicate_ids": int(players["id"].duplicated().sum()),
})
players_quality

rows              1000
null_values          0
duplicate_rows       0
duplicate_ids        0
dtype: int64

In [14]:
# кардинальность колонок
players.nunique(dropna=False).rename("unique_count").to_frame()

,unique_count
id,1000
registration_date,392
registration_type,3
country,6


In [17]:
players["registration_type"].value_counts()

registration_type
vip         364
standard    345
premium     291
Name: count, dtype: int64

In [18]:
players["country"].value_counts()

country
UK    177
RU    169
DE    168
FR    164
US    164
IN    158
Name: count, dtype: int64

In [19]:
# проверка дат
registration_dates = pd.to_datetime(
    players["registration_date"],
    errors="coerce",
)

date_quality = pd.Series(
    {
        "invalid_dates": int(registration_dates.isna().sum()),
        "min_date": registration_dates.min(),
        "max_date": registration_dates.max(),
    }
)
date_quality

invalid_dates                      0
min_date         2023-01-01 00:00:00
max_date         2024-02-29 00:00:00
dtype: object

In [21]:
# проверка id
id_quality = pd.Series({
    "min_id": players["id"].min(),
    "max_id": players["id"].max(),
    "non_positive_ids": int(players["id"].le(0).sum()),
    "is_unique": players["id"].is_unique,
})
id_quality

min_id                 1
max_id              1000
non_positive_ids       0
is_unique           True
dtype: object

# Players итоги
В файле - 1000 строк и 4 колонки
типы колонок которые определил pandas:
id - int64
registration_date - str
registration_type - str
country - str

Уникальность:
id - уникален
registration_date - 392 уникальных значения из 1000 строк
registration_type - 3 уникальных значения на 1000 строк (vip, standard, premium)
country - 6 уникальных значения на 1000 строк (UK, RU, DE, FR, US, IN)

невалидных дат регистарции нет

В ClickHouse можно сделать таблицу примерно такую:
| Column | Type |
| --- | --- |
| id | UInt32 |
| registration_date | Date |
| registration_type | LowCardinality(String) |
| country | LowCardinality(String) |

In [12]:
# Deposits
deposits_path = DATA_DIR / "deposits.csv"

if not deposits_path.is_file():
    raise FileNotFoundError(f"Не найден файл: {deposits_path}")

deposits = pd.read_csv(deposits_path)

In [13]:
deposits.head()

,id,player_id,deposit_date,provider_id,amount,currency
0,1,999,2023-03-14,7,203.45,GBP
1,2,629,2023-07-02,9,711.78,USD
2,3,871,2023-02-19,10,948.56,RUB
3,4,342,2023-09-22,3,744.82,USD
4,5,121,2023-04-04,7,706.98,EUR


In [14]:
deposits.sample(n=5, random_state=42)

,id,player_id,deposit_date,provider_id,amount,currency
1501,1502,304,2023-12-29,6,912.00,RUB
2586,2587,182,2024-01-23,3,369.79,GBP
2653,2654,915,2024-01-30,2,114.88,USD
1055,1056,909,2023-11-19,10,839.63,EUR
705,706,759,2023-04-23,6,931.33,USD


In [15]:
print(f"СТроки: {deposits.shape[0]}")
print(f"Колонки: {deposits.shape[1]}")

СТроки: 5000
Колонки: 6


In [16]:
deposits.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            5000 non-null   int64  
 1   player_id     5000 non-null   int64  
 2   deposit_date  5000 non-null   str    
 3   provider_id   5000 non-null   int64  
 4   amount        5000 non-null   float64
 5   currency      5000 non-null   str    
dtypes: float64(1), int64(3), str(2)
memory usage: 234.5 KB


In [17]:
# проверка качества данных
deposits_quality = pd.Series({
    "rows": len(deposits),
    "null_values": int(deposits.isna().sum().sum()),
    "duplicate_rows": int(deposits.duplicated().sum()),
    "duplicate_ids": int(deposits["id"].duplicated().sum()),
    "unique_players": int(deposits["player_id"].nunique()),
    "unique_providers": int(deposits["provider_id"].nunique()),
    "unique_currencies": int(deposits["currency"].nunique()),
})
deposits_quality

rows                 5000
null_values             0
duplicate_rows          0
duplicate_ids           0
unique_players        994
unique_providers       10
unique_currencies       4
dtype: int64

In [18]:
# кардинальность колонок
deposits.nunique(dropna=False).rename("unique_count").to_frame()

,unique_count
id,5000
player_id,994
deposit_date,425
provider_id,10
amount,4855
currency,4


In [19]:
# проверка id
deposit_id_quality = pd.Series({
    "min_id": deposits["id"].min(),
    "max_id": deposits["id"].max(),
    "non_positive_ids": int(deposits["id"].le(0).sum()),
    "is_unique": deposits["id"].is_unique,
})
deposit_id_quality

min_id                 1
max_id              5000
non_positive_ids       0
is_unique           True
dtype: object

In [20]:
# проверка связи с игроками
unknown_player_ids = deposits.loc[
    ~deposits["player_id"].isin(players["id"]),
    ["id", "player_id"],
]
unknown_player_ids

,id,player_id


In [ ]:
# проверка дат
deposit_dates = pd.to_datetime(
    deposits["deposit_date"],
    errors="coerce",
)
deposit_date_quality = pd.Series({
    "invalid_dates": int(deposit_dates.isna().sum()),
    "min_date": deposit_dates.min(),
    "max_date": deposit_dates.max(),
    "unique_dates": int(deposit_dates.nunique()),
})
deposit_date_quality

invalid_dates                      0
min_date         2023-01-01 00:00:00
max_date         2024-02-29 00:00:00
unique_dates                     425
dtype: object

In [ ]:
# проверка amount
amount_quality = pd.Series({
    "null_amounts": int(deposits["amount"].isna().sum()),
    "zero_amounts": int(deposits["amount"].eq(0).sum()),
    "negative_amounts": int(deposits["amount"].lt(0).sum()),
    "min_amount": deposits["amount"].min(),
    "max_amount": deposits["amount"].max(),
})
amount_quality

null_amounts          0.00
zero_amounts          0.00
negative_amounts      0.00
min_amount           10.19
max_amount          999.68
dtype: float64

In [25]:
# валюты
deposits["currency"].value_counts().rename("deposit_count").to_frame()

,deposit_count
currency,
RUB,1267
EUR,1267
GBP,1237
USD,1229


In [26]:
# провайдеры
deposits["provider_id"].value_counts().sort_index().rename("deposit_count").to_frame()

,deposit_count
provider_id,
1,512
2,507
3,539
4,498
5,490
6,481
7,467
8,473
9,520


# Deposits итоги
В файле - 5000 строк и 6 колонки
типы колонок которые определил pandas:
id - int64  
player_id - int64  
deposit_date - str    
provider_id - int64  
amount - float64
currency - str

нет несуществующих игроков

Уникальность:
id - уникален
currency - 4 уникальных значения на 5000 строк (RUB, EUR, GBP, USD)

невалидных дат платежей нет

В ClickHouse можно сделать таблицу примерно такую:
| Column | Type |
| --- | --- |
| id | UInt32 |
| player_id | UInt32 |
| deposit_date | Date |
| provider_id | UInt32 |
| amount | Decimal(10, 2) |
| country | LowCardinality(String) |

In [ ]:
# withdrawals
withdrawals_path = DATA_DIR / "withdrawals.csv"

if not withdrawals_path.is_file():
    raise FileNotFoundError(f"Не найден файл: {withdrawals_path}")

withdrawals = pd.read_csv(withdrawals_path)

In [28]:
withdrawals.head()

,id,player_id,withdrawal_date,provider_id,amount,currency
0,1,3,2024-02-20,1,885.51,USD
1,2,903,2023-08-12,1,59.76,USD
2,3,525,2023-08-31,3,222.66,USD
3,4,462,2024-02-25,1,452.18,RUB
4,5,754,2023-06-12,1,189.44,RUB


In [29]:
withdrawals.sample(n=5, random_state=42)

,id,player_id,withdrawal_date,provider_id,amount,currency
1801,1802,880,2023-10-05,2,70.94,USD
1190,1191,318,2024-01-23,10,657.44,GBP
1817,1818,174,2023-10-21,6,857.55,GBP
251,252,926,2023-04-03,9,304.68,EUR
2505,2506,974,2023-01-03,4,770.74,EUR


In [30]:
withdrawals.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               3000 non-null   int64  
 1   player_id        3000 non-null   int64  
 2   withdrawal_date  3000 non-null   str    
 3   provider_id      3000 non-null   int64  
 4   amount           3000 non-null   float64
 5   currency         3000 non-null   str    
dtypes: float64(1), int64(3), str(2)
memory usage: 140.8 KB


In [31]:
print(f"СТроки: {withdrawals.shape[0]}")
print(f"Колонки: {withdrawals.shape[1]}")

СТроки: 3000
Колонки: 6


In [32]:
withdrawals_quality = pd.Series(
    {
        "rows": len(withdrawals),
        "null_values": int(withdrawals.isna().sum().sum()),
        "duplicate_rows": int(withdrawals.duplicated().sum()),
        "duplicate_ids": int(withdrawals["id"].duplicated().sum()),
        "unique_players": int(withdrawals["player_id"].nunique()),
        "unique_providers": int(withdrawals["provider_id"].nunique()),
        "unique_currencies": int(withdrawals["currency"].nunique()),
    },
    name="value",
)
withdrawals_quality

rows                 3000
null_values             0
duplicate_rows          0
duplicate_ids           0
unique_players        936
unique_providers       10
unique_currencies       4
Name: value, dtype: int64

In [33]:
withdrawal_id_quality = pd.Series(
    {
        "min_id": withdrawals["id"].min(),
        "max_id": withdrawals["id"].max(),
        "non_positive_ids": int(
            withdrawals["id"].le(0).sum()
        ),
        "is_unique": withdrawals["id"].is_unique,
    },
    name="value",
)
withdrawal_id_quality

min_id                 1
max_id              3000
non_positive_ids       0
is_unique           True
Name: value, dtype: object

In [34]:
unknown_withdrawal_players = withdrawals.loc[
    ~withdrawals["player_id"].isin(players["id"]),
    ["id", "player_id"],
]
unknown_withdrawal_players

,id,player_id


In [35]:
withdrawal_dates = pd.to_datetime(
    withdrawals["withdrawal_date"],
    errors="coerce",
)
withdrawal_date_quality = pd.Series(
    {
        "invalid_dates": int(withdrawal_dates.isna().sum()),
        "min_date": withdrawal_dates.min(),
        "max_date": withdrawal_dates.max(),
        "unique_dates": int(withdrawal_dates.nunique()),
    },
    name="value",
)
withdrawal_date_quality

invalid_dates                      0
min_date         2023-01-01 00:00:00
max_date         2024-02-29 00:00:00
unique_dates                     425
Name: value, dtype: object

In [37]:
withdrawal_amount_quality = pd.Series(
    {
        "null_amounts": int(
            withdrawals["amount"].isna().sum()
        ),
        "zero_amounts": int(
            withdrawals["amount"].eq(0).sum()
        ),
        "negative_amounts": int(
            withdrawals["amount"].lt(0).sum()
        ),
        "min_amount": withdrawals["amount"].min(),
        "max_amount": withdrawals["amount"].max(),
    },
    name="value",
)
withdrawal_amount_quality

null_amounts          0.00
zero_amounts          0.00
negative_amounts      0.00
min_amount           10.05
max_amount          999.98
Name: value, dtype: float64

In [38]:
withdrawals["currency"].value_counts().rename("withdrawal_count").to_frame()

,withdrawal_count
currency,
USD,787
GBP,746
EUR,739
RUB,728


# Withdrawals итоги
В файле - 3000 строк и 6 колонки
типы колонок которые определил pandas:
id - int64  
player_id - int64  
withdrawal_date - str    
provider_id - int64  
amount - float64
currency - str

нет null значений
нет несуществующих игроков

Уникальность:
id - уникален
currency - 4 уникальных значения на 3000 строк (RUB, EUR, GBP, USD)

невалидных дат нет

В ClickHouse можно сделать таблицу примерно такую:
| Column | Type |
| --- | --- |
| id | UInt32 |
| player_id | UInt32 |
| withdrawal_date | Date |
| provider_id | UInt32 |
| amount | Decimal(10, 2) |
| country | LowCardinality(String) |

In [ ]:
# games
games_path = DATA_DIR / "games.csv"

if not games_path.is_file():
    raise FileNotFoundError(f"Не найден файл: {games_path}")

games = pd.read_csv(games_path)

In [40]:
games.head()

,id,player_id,game_date,amount,currency,provider_id,game_id
0,1,523,2023-05-06,36.56,USD,4,6
1,2,705,2023-12-17,397.19,GBP,5,14
2,3,504,2023-09-23,194.37,USD,2,42
3,4,207,2023-11-06,378.52,GBP,7,15
4,5,161,2023-08-12,28.01,EUR,2,36


In [41]:
games.sample(n=5, random_state=42)

,id,player_id,game_date,amount,currency,provider_id,game_id
6500,6501,691,2023-03-21,373.37,USD,3,36
2944,2945,945,2024-01-23,429.81,USD,6,28
2024,2025,775,2023-12-05,484.72,USD,3,39
263,264,599,2024-01-29,455.24,RUB,7,50
4350,4351,761,2024-02-23,453.39,GBP,4,36


In [42]:
games.info()

<class 'pandas.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           7000 non-null   int64  
 1   player_id    7000 non-null   int64  
 2   game_date    7000 non-null   str    
 3   amount       7000 non-null   float64
 4   currency     7000 non-null   str    
 5   provider_id  7000 non-null   int64  
 6   game_id      7000 non-null   int64  
dtypes: float64(1), int64(4), str(2)
memory usage: 382.9 KB


In [43]:
print(f"СТроки: {games.shape[0]}")
print(f"Колонки: {games.shape[1]}")

СТроки: 7000
Колонки: 7


In [44]:
games_quality = pd.Series(
    {
        "rows": len(games),
        "null_values": int(games.isna().sum().sum()),
        "duplicate_rows": int(games.duplicated().sum()),
        "duplicate_ids": int(games["id"].duplicated().sum()),
        "unique_players": int(games["player_id"].nunique()),
        "unique_providers": int(games["provider_id"].nunique()),
        "unique_games": int(games["game_id"].nunique()),
        "unique_currencies": int(games["currency"].nunique()),
    },
    name="value",
)
games_quality


rows                 7000
null_values             0
duplicate_rows          0
duplicate_ids           0
unique_players       1000
unique_providers       10
unique_games           50
unique_currencies       4
Name: value, dtype: int64

In [45]:
game_operation_id_quality = pd.Series(
    {
        "min_id": games["id"].min(),
        "max_id": games["id"].max(),
        "non_positive_ids": int(games["id"].le(0).sum()),
        "is_unique": games["id"].is_unique,
    },
    name="value",
)
game_operation_id_quality

min_id                 1
max_id              7000
non_positive_ids       0
is_unique           True
Name: value, dtype: object

In [46]:
unknown_game_players = games.loc[
    ~games["player_id"].isin(players["id"]),
    ["id", "player_id"],
]
unknown_game_players

,id,player_id


In [47]:
game_dates = pd.to_datetime(
    games["game_date"],
    errors="coerce",
)
game_date_quality = pd.Series(
    {
        "invalid_dates": int(game_dates.isna().sum()),
        "min_date": game_dates.min(),
        "max_date": game_dates.max(),
        "unique_dates": int(game_dates.nunique()),
    },
    name="value",
)
game_date_quality

invalid_dates                      0
min_date         2023-01-01 00:00:00
max_date         2024-02-29 00:00:00
unique_dates                     425
Name: value, dtype: object

In [49]:
game_amount_quality = pd.Series(
    {
        "null_amounts": int(games["amount"].isna().sum()),
        "zero_amounts": int(games["amount"].eq(0).sum()),
        "negative_amounts": int(games["amount"].lt(0).sum()),
        "min_amount": games["amount"].min(),
        "max_amount": games["amount"].max(),
    },
    name="value",
)
game_amount_quality

null_amounts          0.00
zero_amounts          0.00
negative_amounts      0.00
min_amount            1.24
max_amount          499.97
Name: value, dtype: float64

In [50]:
games["currency"].value_counts().rename("game_operations_count").to_frame()

,game_operations_count
currency,
GBP,1795
EUR,1771
RUB,1754
USD,1680


# Games итоги
В файле - 7000 строк и 7 колонки
типы колонок которые определил pandas:
id - int64  
player_id - int64  
game_date - str    
amount - float64
currency - str
provider_id - int64  
game_id - int64

нет null значений
нет несуществующих игроков

Уникальность:
id - уникален
currency - 4 уникальных значения на 7000 строк (RUB, EUR, GBP, USD)

невалидных дат игр нет

В ClickHouse можно сделать таблицу примерно такую:
| Column | Type |
| --- | --- |
| id | UInt32 |
| player_id | UInt32 |
| game_date | Date |
| amount | Decimal(10, 2) |
| country | LowCardinality(String) |
| provider_id | UInt32 |
| game_id | UInt32 |

In [ ]:
# currency_rates
currency_rates_path = DATA_DIR / "currency_rates.csv"

if not currency_rates_path.is_file():
    raise FileNotFoundError(f"Не найден файл: {currency_rates_path}")

currency_rates = pd.read_csv(currency_rates_path)

In [52]:
currency_rates.head()

,date,currency,rate_to_usd
0,2023-01-01,USD,1.0000
1,2023-01-01,EUR,1.1565
2,2023-01-01,GBP,1.2227
3,2023-01-01,RUB,68.7021
4,2023-01-02,USD,1.0000


In [53]:
currency_rates.sample(n=5, random_state=42)

,date,currency,rate_to_usd
1046,2023-09-19,GBP,1.1818
745,2023-07-06,EUR,1.0694
785,2023-07-16,EUR,0.8697
367,2023-04-02,RUB,83.8223
1029,2023-09-15,EUR,1.0747


In [54]:
currency_rates.info()

<class 'pandas.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         1704 non-null   str    
 1   currency     1704 non-null   str    
 2   rate_to_usd  1704 non-null   float64
dtypes: float64(1), str(2)
memory usage: 40.1 KB


In [56]:
print(f"СТроки: {currency_rates.shape[0]}")
print(f"Колонки: {currency_rates.shape[1]}")

СТроки: 1704
Колонки: 3


In [58]:
currency_rates_quality = pd.Series(
    {
        "rows": len(currency_rates),
        "null_values": int(currency_rates.isna().sum().sum()),
        "duplicate_rows": int(currency_rates.duplicated().sum()),
        "duplicate_date_currency": int(
            currency_rates.duplicated(
                subset=["date", "currency"]
            ).sum()
        ),
        "unique_dates": int(currency_rates["date"].nunique()),
        "unique_currencies": int(currency_rates["currency"].nunique()),
    },
    name="value",
)
currency_rates_quality

rows                       1704
null_values                   0
duplicate_rows                0
duplicate_date_currency       0
unique_dates                426
unique_currencies             4
Name: value, dtype: int64

In [59]:
rate_dates = pd.to_datetime(
    currency_rates["date"],
    errors="coerce",
)
rate_date_quality = pd.Series(
    {
        "invalid_dates": int(rate_dates.isna().sum()),
        "min_date": rate_dates.min(),
        "max_date": rate_dates.max(),
        "unique_dates": int(rate_dates.nunique()),
    },
    name="value",
)
rate_date_quality

invalid_dates                      0
min_date         2023-01-01 00:00:00
max_date         2024-03-01 00:00:00
unique_dates                     426
Name: value, dtype: object

In [60]:
currency_rates["currency"].value_counts().rename("rates_count").to_frame()

,rates_count
currency,
USD,426
EUR,426
GBP,426
RUB,426


In [61]:
rates_per_currency = (
    currency_rates
    .groupby("currency", as_index=False)
    .agg(
        rates_count=("date", "count"),
        unique_dates=("date", "nunique"),
    )
)
rates_per_currency

,currency,rates_count,unique_dates
0,EUR,426,426
1,GBP,426,426
2,RUB,426,426
3,USD,426,426


In [62]:
rate_quality = pd.Series(
    {
        "null_rates": int(
            currency_rates["rate_to_usd"].isna().sum()
        ),
        "zero_rates": int(
            currency_rates["rate_to_usd"].eq(0).sum()
        ),
        "negative_rates": int(
            currency_rates["rate_to_usd"].lt(0).sum()
        ),
        "min_rate": currency_rates["rate_to_usd"].min(),
        "max_rate": currency_rates["rate_to_usd"].max(),
    },
    name="value",
)
rate_quality

null_rates         0.0000
zero_rates         0.0000
negative_rates     0.0000
min_rate           0.7003
max_rate          89.9935
Name: value, dtype: float64

In [65]:
rate_statistics = (
    currency_rates
    .groupby("currency")["rate_to_usd"]
    .agg(["min", "max", "mean"])
    .round(4)
)
rate_statistics

,min,max,mean
currency,,,
EUR,0.8009,1.1991,0.9860
GBP,0.7003,1.4998,1.1217
RUB,60.0958,89.9935,75.2902
USD,1.0000,1.0000,1.0000


# Currency Rate итоги
В файле - 1704 строк и 3 колонки
типы колонок которые определил pandas: 
date - str    
currency - str
rate_to_usd - float64

нет null значений
Показывает что фактически курс хранится как количество единиц валют за один USD
значит формула перевода будет amount_in_usd = amount / rate_to_usd

Уникальность:
currency - 4 уникальных значения на 1704 строк (RUB, EUR, GBP, USD)

невалидных дат нет

В ClickHouse можно сделать таблицу примерно такую:
| Column | Type |
| --- | --- |
| date | Date |
| country | LowCardinality(String) |
| rate_to_usd | Decimal(10, 4) |

In [66]:
# providers_map
providers_map_path = DATA_DIR / "providers_map.csv"

if not providers_map_path.is_file():
    raise FileNotFoundError(f"Не найден файл: {providers_map_path}")

providers_map = pd.read_csv(providers_map_path)

In [67]:
providers_map.head()

,id,provider_name
0,1,Provider_1
1,2,Provider_2
2,3,Provider_3
3,4,Provider_4
4,5,Provider_5


In [68]:
providers_map.sample(n=5, random_state=42)

,id,provider_name
8,9,Provider_9
1,2,Provider_2
5,6,Provider_6
0,1,Provider_1
7,8,Provider_8


In [71]:
print(f"СТроки: {providers_map.shape[0]}")
print(f"Колонки: {providers_map.shape[1]}")

СТроки: 10
Колонки: 2


In [69]:
providers_map.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             10 non-null     int64
 1   provider_name  10 non-null     str  
dtypes: int64(1), str(1)
memory usage: 292.0 bytes


In [72]:
providers_quality = pd.Series(
    {
        "rows": len(providers_map),
        "null_values": int(providers_map.isna().sum().sum()),
        "duplicate_rows": int(providers_map.duplicated().sum()),
        "duplicate_ids": int(providers_map["id"].duplicated().sum()),
        "duplicate_names": int(
            providers_map["provider_name"].duplicated().sum()
        ),
    },
    name="value",
)
providers_quality

rows               10
null_values         0
duplicate_rows      0
duplicate_ids       0
duplicate_names     0
Name: value, dtype: int64

In [73]:
provider_id_quality = pd.Series(
    {
        "min_id": providers_map["id"].min(),
        "max_id": providers_map["id"].max(),
        "non_positive_ids": int(
            providers_map["id"].le(0).sum()
        ),
        "is_unique": providers_map["id"].is_unique,
    },
    name="value",
)
provider_id_quality

min_id                 1
max_id                10
non_positive_ids       0
is_unique           True
Name: value, dtype: object

In [74]:
empty_provider_names = (
    providers_map["provider_name"]
    .str.strip()
    .eq("")
    .sum()
)
empty_provider_names

np.int64(0)

In [75]:
providers_map.sort_values("id")

,id,provider_name
0,1,Provider_1
1,2,Provider_2
2,3,Provider_3
3,4,Provider_4
4,5,Provider_5
5,6,Provider_6
6,7,Provider_7
7,8,Provider_8
8,9,Provider_9
9,10,Provider_10


# Providers map итоги
В файле - 10 строк и 2 колонки
типы колонок которые определил pandas: 
id - int64    
provider_name - str

нет null значений

Уникальность:
id уникален

пустых названий нет

В ClickHouse можно сделать таблицу примерно такую:
| Column | Type |
| --- | --- |
| id | UInt32 |
| provider_name | String |

In [76]:
def count_unknown_references(
    fact: pd.DataFrame,
    fact_key: str,
    dimension: pd.DataFrame,
    dimension_key: str,
) -> int:
    return int(
        (~fact[fact_key].isin(dimension[dimension_key])).sum()
    )

provider_reference_quality = pd.Series(
    {
        "unknown_deposit_providers": count_unknown_references(
            deposits,
            "provider_id",
            providers_map,
            "id",
        ),
        "unknown_withdrawal_providers": count_unknown_references(
            withdrawals,
            "provider_id",
            providers_map,
            "id",
        ),
        "unknown_game_providers": count_unknown_references(
            games,
            "provider_id",
            providers_map,
            "id",
        ),
    },
    name="value",
)
provider_reference_quality

unknown_deposit_providers       0
unknown_withdrawal_providers    0
unknown_game_providers          0
Name: value, dtype: int64

In [77]:
# games_map
games_map_path = DATA_DIR / "games_map.csv"

if not games_map_path.is_file():
    raise FileNotFoundError(f"Не найден файл: {games_map_path}")

games_map = pd.read_csv(games_map_path)

In [78]:
games_map.head()

,id,game_name,provider_id
0,1,Game_1,7
1,2,Game_2,6
2,3,Game_3,1
3,4,Game_4,2
4,5,Game_5,8


In [79]:
games_map.sample(n=5, random_state=42)

,id,game_name,provider_id
13,14,Game_14,10
39,40,Game_40,9
30,31,Game_31,9
45,46,Game_46,4
17,18,Game_18,4


In [80]:
games_map.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           50 non-null     int64
 1   game_name    50 non-null     str  
 2   provider_id  50 non-null     int64
dtypes: int64(2), str(1)
memory usage: 1.3 KB


In [81]:
print(f"СТроки: {games_map.shape[0]}")
print(f"Колонки: {games_map.shape[1]}")

СТроки: 50
Колонки: 3


In [83]:
games_map_quality = pd.Series(
    {
        "rows": len(games_map),
        "null_values": int(games_map.isna().sum().sum()),
        "duplicate_rows": int(games_map.duplicated().sum()),
        "duplicate_ids": int(games_map["id"].duplicated().sum()),
        "duplicate_names": int(
            games_map["game_name"].duplicated().sum()
        ),
        "unique_providers": int(
            games_map["provider_id"].nunique()
        ),
    },
    name="value",
)
games_map_quality

rows                50
null_values          0
duplicate_rows       0
duplicate_ids        0
duplicate_names      0
unique_providers    10
Name: value, dtype: int64

In [84]:
game_map_id_quality = pd.Series(
    {
        "min_id": games_map["id"].min(),
        "max_id": games_map["id"].max(),
        "non_positive_ids": int(
            games_map["id"].le(0).sum()
        ),
        "is_unique": games_map["id"].is_unique,
    },
    name="value",
)
game_map_id_quality

min_id                 1
max_id                50
non_positive_ids       0
is_unique           True
Name: value, dtype: object

In [85]:
empty_game_names = (
    games_map["game_name"]
    .str.strip()
    .eq("")
    .sum()
)
empty_game_names

np.int64(0)

In [86]:
unknown_game_map_providers = games_map.loc[
    ~games_map["provider_id"].isin(providers_map["id"]),
    ["id", "game_name", "provider_id"],
]
unknown_game_map_providers

,id,game_name,provider_id


In [87]:
unknown_game_ids = games.loc[
    ~games["game_id"].isin(games_map["id"]),
    ["id", "game_id"],
]
unknown_game_ids

,id,game_id


In [91]:
games_with_mapping = games.merge(
    games_map[["id", "provider_id"]],
    how="left",
    left_on="game_id",
    right_on="id",
    suffixes=("_operation", "_mapping"),
    validate="many_to_one",
)
provider_mismatches = games_with_mapping.loc[
    games_with_mapping["provider_id_operation"]
    != games_with_mapping["provider_id_mapping"]
]
provider_mismatches

,id_operation,player_id,game_date,amount,currency,provider_id_operation,game_id,id_mapping,provider_id_mapping
0,1,523,2023-05-06,36.56,USD,4,6,6,3
1,2,705,2023-12-17,397.19,GBP,5,14,14,10
2,3,504,2023-09-23,194.37,USD,2,42,42,7
3,4,207,2023-11-06,378.52,GBP,7,15,15,5
4,5,161,2023-08-12,28.01,EUR,2,36,36,5
...,...,...,...,...,...,...,...,...,...
6995,6996,897,2023-04-18,9.54,RUB,10,24,24,2
6996,6997,948,2023-01-06,152.19,EUR,3,5,5,8
6997,6998,895,2023-08-13,163.63,RUB,6,15,15,5
6998,6999,546,2023-06-13,267.74,GBP,9,7,7,4


# games map итоги
В файле - 50 строк и 3 колонки
типы колонок которые определил pandas: 
id - int64    
game_name - str
provider_id - int64

нет null значений

Уникальность:
id уникален

пустых названий нет
нет невалидных провайдеров
для игр итоговго из games map несовпадает 6322 строки из 7000
для витрины в целом эти поля не нужны, но в целом очень странно

В ClickHouse можно сделать таблицу примерно такую:
| Column | Type |
| --- | --- |
| id | UInt32 |
| game_name | String |
| provider_id | String |